# Makemore Part 3: Activations, Gradients & Batch Norm

**Bedtime Study Guide Version**

This notebook contains the code from Andrej Karpathy's `makemore` Part 3, annotated with narrative notes explaining the "Why" behind the "How".

---

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

In [3]:
len(words)

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

In [5]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%


## Theory Block 1: Initialization Problems

### Topic 1: The "Hockey Stick" Loss
**Context:** When we first start training, we usually see the loss start very high, drop massively in the first few iterations, and then plateau. The graph looks like a hockey stick.

* **The Problem:** The model is "surprised" by the training data at initialization. 
    * At the start, the model assigns random probability to every character. Since there are 27 characters, the probability of any character should be roughly $1/27$.
    * Loss is $- \log(\text{prob})$. So expected initial loss should be $- \log(1/27) \approx 3.29$.
    * In the video, the initial loss is much higher (e.g., 20+). This means the model is confidently predicting the *wrong* characters at initialization.
* **The Fix:** **Better Weight Initialization.**
    * We want the logits (outputs of the last layer) to be close to 0 initially, so probabilities are uniform.
    * **Solution:** Multiply the initial weights `W2` by a small number (e.g., `0.01`) and set the bias `b2` to `0`.

### Topic 2: The Saturated Tanh (Vanishing Gradients)
**Context:** We use `tanh` as our activation function in the hidden layer. It squashes numbers between -1 and 1.

* **The Problem:** **Dead Neurons.**
    * If the weights entering the `tanh` are too big, the output will be exactly -1.0 or 1.0.
    * **Why is this bad?** Recall `micrograd`: The derivative of `tanh(x)` involves $(1 - t^2)$. If $t$ is 1 or -1, the gradient is **zero**.
    * If the gradient is zero, backpropagation stops. No information flows backward to update the weights. That neuron is "dead"—it learns nothing.
* **The Fix:** **Scale down the weights.**
    * Similar to Topic 1, we multiply the inner layer weights `W1` by a small factor (e.g., `0.1` or `0.2`).
    * *Result:* The `tanh` outputs are now distributed nicely between -1 and 1 (mostly around 0). Gradients can flow!

### Topic 3: Kaiming Initialization (The Math-y Fix)
**Context:** We fixed Topic 2 by multiplying by `0.2` or `0.1`. But that was a guess. If the network is deeper, `0.1` might be too small (signal vanishes) or too big (signal explodes).

* **The Problem:** We shouldn't have to guess "magic numbers" for initialization. We need a rule that works for any layer size.
* **The Fix:** **Kaiming Initialization (He Init).**
    * The goal: We want the variance of the outputs to be the same as the variance of the inputs (Unit Gaussian).
    * Mathematically, if you have `fan_in` inputs, you should scale weights by $\sqrt{\frac{2}{\text{fan\_in}}}$ (if using ReLU) or $\frac{5}{3} \cdot \frac{1}{\sqrt{\text{fan\_in}}}$ (if using Tanh).
    * *Result:* You don't need to guess `0.1` anymore. The math guarantees the signal stays healthy deep into the network.

In [6]:
# MLP revisited
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5) #* 0.2
#b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.randn(vocab_size,                      generator=g) * 0

# BatchNorm parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

## Theory Block 2: Batch Normalization

### Topic 4: Batch Normalization (The Big Gun)
**Context:** Even with Kaiming Init, deep networks are fragile. If one layer shifts a little, the next layer freaks out. This is "Internal Covariate Shift."

* **The Problem:** We want the pre-activations (inputs to `tanh`) to be Gaussian (mean 0, std 1) so `tanh` doesn't saturate. It's annoying to perfectly tune weights to keep them Gaussian.
* **The Fix:** **Just force them to be Gaussian.**
    * **Step 1:** Calculate the `mean` and `std` of the current batch.
    * **Step 2:** Normalize the batch: $x_{new} = \frac{x - \text{mean}}{\text{std}}$. Now it's perfectly Gaussian!
    * **Step 3 (Crucial):** What if we *don't* want them to be perfect Gaussian? (Maybe the network *wants* to be shifted). We multiply by a learned `gamma` (scale) and add `beta` (shift).
    * *Result:* The network is incredibly stable. You can be sloppy with initialization, and Batch Norm fixes it for you.

* **The "Inference" Problem:**
    * Batch Norm works on *batches*. But at test time (inference), we might pass in a single example. We can't calculate a "mean" of one item.
    * **The Fix:** During training, we keep a "running average" (exponential moving average) of the mean and std. At test time, we use this running average instead of the batch statistics.

In [7]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hpreact = embcat @ W1 #+ b1 # hidden layer pre-activation
  # BatchNorm layer
  # -------------------------------------------------------------
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
  # -------------------------------------------------------------
  # Non-linearity
  h = torch.tanh(hpreact) # hidden layer
  logits = h @ W2 + b2 # output layer
  loss = F.cross_entropy(logits, Yb) # loss function
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  

In [8]:
plt.plot(lossi)

In [9]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 # + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)


In [10]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 # + b1
  #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

## Theory Block 3: Code Structure

### Topic 5: PyTorch-ifying the Code
**Context:** Our code is messy script-like spaghetti.

* **The Problem:** It's hard to build deeper networks (adding layers requires copy-pasting code).
* **The Fix:** **Classes and Containers.**
    * Karpathy refactors the code to look exactly like PyTorch.
    * He creates classes: `Linear`, `BatchNorm1d`, `Tanh`, `Embedding`.
    * Each class has a `__call__` (forward pass) and a list of `parameters()`.
    * He puts them in a `Sequential` container.
    * *Result:* The training loop becomes clean: `model(x)`, `loss.backward()`, `optimizer.step()`.

In [13]:
# Let's train a deeper network
# The classes we create here are the same API as nn.Module in PyTorch

class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]
# layers = [
#   Linear(n_embd * block_size, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size),
# ]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

In [14]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization

## Theory Block 4: Evaluating Network Health

### Topic 6: Diagnostics (Visualizations)
**Context:** How do we know if our network is healthy?

* **The Visualization:**
    * **Activation Histogram:** Plot the output of `tanh`. If it looks like two tall bars at -1 and 1, your weights are too big (saturation). It should look like a nice hill.
    * **Gradient Histogram:** Plot the gradients. If they are all zero, your neurons are dead.
    * **Update-to-Data Ratio:** Measure `update_size / param_value`.
        * If the ratio is `1e-3` (0.001), it's good.
        * If it's too low, the model isn't learning (learning rate too low).
        * If it's too high, the weights are trashing around (learning rate too high).

In [15]:
# visualize histograms
plt.figure(figsize=(20, 4))
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__})')
plt.legend(legends);
plt.title('activation distribution');

In [16]:
# visualize histograms
plt.figure(figsize=(20, 4))
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__})')
plt.legend(legends);
plt.title('gradient distribution');

In [17]:
# visualize histograms
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends);
plt.title('weights gradient distribution');

In [18]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);
plt.title('update data ratio');